# F01. The tokenizer, in C

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/cpython-internals/blob/main/lessons/f01-the-tokenizer-in-c/f01.ipynb)

Your file is on disk. The tokenizer has to turn it into [tokens](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#token), and the honest question is how much of your file it is holding while it does that.

The answer is one line. Not the file, not a window, one line, and the lines before it are already gone. Almost everything else about the C tokenizer follows from that.

![four pointers into a single line of source, with the rest of the file not read yet](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/f01-the-tokenizer-in-c/diagrams/one-line-in-the-buffer.svg)

## About the source references

Now and then this lesson points at CPython's own source, like this: `Parser/lexer/state.h:6-8@v3.15.0rc1`.

Read it as three parts: the file, the lines, and the release those line numbers belong to. Sometimes there is a fourth part after a `#`, which is the name of the thing those lines are inside.

Every reference is a link, and every one is checked against the pinned source on each change, so a stale reference fails the build instead of sending you somewhere wrong. You never have to read any of it. The references are there so you can go deeper when you want to, and so you can check that this lesson is not making things up.

## Setup

Colab does not come with the small package these lessons use, so the next cell installs it. If you are running this from a checkout of the repository it is already installed and the cell does nothing.

In [ ]:
import sys

if sys.version_info < (3, 14):
    print("This lesson needs CPython 3.14 or newer.")
    print(f"This runtime is {sys.version.split()[0]}, and the cells below will not run on it.")
else:
    try:
        import pyxray
    except ImportError:
        %pip install -q "pyxray @ git+https://github.com/tamnd/cpython-internals@main#subdirectory=pyxray"
        import pyxray

## Which Python is this

Everything below was checked against the version this cell prints and against 3.14. Where the two disagree, the lesson says so.

In [ ]:
import pyxray

pyxray.show()

## The token set is a table

Start with the numbers. `NAME` is 1, `NUMBER` is 2, and something has to decide that.

Nothing in the C decides it. `Grammar/Tokens` is a plain text file of one token per line, and the order of the lines is the numbering. A script reads it and writes four files, one of which is the header the [tokenizer](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#tokenizer) includes and another of which is `Lib/token.py`, the module you have been importing.

![the four files written from Grammar/Tokens and who reads each one](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/f01-the-tokenizer-in-c/diagrams/one-table-five-files.svg)

Look at what that buys. The C and the Python cannot drift apart on what `NUMBER` means, because neither of them is where the answer lives. the token numbers in Lib/token.py are the line order of Grammar/Tokens, and the file says at the top that it was generated and the module admits it in its second line.

In [ ]:
import inspect
import token

print(inspect.getsource(token).splitlines()[1])
print()
print(f"names in tok_name:          {len(token.tok_name)}")
print(f"tokens with a spelling:     {len(token.EXACT_TOKEN_TYPES)}")
print()
for number in range(7):
    print(f"  {number:2}  {token.tok_name[number]}")

Those first seven are the ones with no spelling: you cannot write an `INDENT` the way you write a `+`. They are lines 6 to 12 of `Grammar/Tokens`, in that order.

The rest of the file has two columns. `LPAR '('` says the token is called `LPAR` and it is spelled `(`. That second column becomes `EXACT_TOKEN_TYPES`, a dict from the spelling to the number, and it is the reason every operator arrives as the single type OP and the exact spelling is recovered afterwards from a table, rather than the tokenizer having a separate type for each one.

In [ ]:
import io
import tokenize

for found in tokenize.generate_tokens(io.StringIO("a += b @ c\n").readline):
    if found.type == token.OP:
        kind = token.tok_name[found.type]
        exact = token.tok_name[found.exact_type]
        print(f"  {found.string:4}  type={kind:4}  exact_type={exact}")

`+=` and `@` come back as `OP`, and `exact_type` is a lookup in that generated dict. The C tokenizer does the same thing in the same place, from the same table.

If you ever add an operator to Python, this is the file you edit. The comment on the first four lines of `Grammar/Tokens` is there because one place does not update itself: the PEG generator has its own copy, and it has to be edited by hand.

## Four ways in, and then the same lexer

Text reaches the tokenizer four ways, and the whole interface is a ten line header. [Parser/tokenizer/tokenizer.h:6-10@v3.15.0rc1](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/tokenizer/tokenizer.h#L6-L10) declares four constructors, one per kind of input.

![the four tokenizer constructors, where each gets its text and who calls it](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/f01-the-tokenizer-in-c/diagrams/four-front-ends.svg)

What is nice about this is how little they differ. Each one fills in a `struct tok_state` and sets one field, its [underflow](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#underflow) function, to its own idea of how to get the next line. After that the lexer never asks where the text came from. When it runs out it calls that function pointer, at [Parser/lexer/lexer.c:74-82@v3.15.0rc1](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/lexer/lexer.c#L74-L82), and carries on.

Two of the four are one line apart in the same `if`. [Parser/pegen.c:1055-1060@v3.15.0rc1](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/pegen.c#L1055-L1060) picks `_PyTokenizer_FromUTF8` when the caller set `PyCF_IGNORE_COOKIE` and `_PyTokenizer_FromString` when it did not, and the caller that sets it is `compile` when you hand it a `str`. So compiling the same source as bytes and as a str goes through two different tokenizer constructors, and the difference shows up as whether a coding cookie is obeyed or ignored.

In [ ]:
COOKIE = b"# -*- coding: ascii -*-\nname = 'caf\xe9'\n"


def compiles(source):
    try:
        compile(source, "<here>", "exec")
    except SyntaxError as unhappy:
        return f"{type(unhappy).__name__}: {unhappy.msg}"
    return "compiles fine"


print("as bytes: ", compiles(COOKIE))
print("as a str: ", compiles(COOKIE.decode("latin-1")))

Same characters, same [cookie](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#coding-cookie), two answers. As bytes the cookie is a promise about the encoding and the tokenizer holds you to it. As a str the bytes question is already settled, so the cookie is treated as a comment and skipped. One `if` in `pegen.c`, two constructors, and a difference you can see from Python.

The other two are `_PyTokenizer_FromFile`, which [Parser/pegen.c:999@v3.15.0rc1](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/pegen.c#L999) uses when you run a script, and `_PyTokenizer_FromReadline`, which [Python/Python-tokenize.c:69@v3.15.0rc1](https://github.com/python/cpython/blob/v3.15.0rc1/Python/Python-tokenize.c#L69) uses for the `tokenize` module. Every token you looked at in T02 came through that last one.

## It really is one line

Back to the opening claim. A [debug build](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#debug-build) compiled with `-d` prints a line to stderr every time the tokenizer runs dry and calls `underflow`, and what it prints is the entire contents of the buffer at that moment, plus the value of `tok->done`.

the tokenizer holds one line of your file at a time, and a file that fails to parse gets read more than once

How much of your file is in the tokenizer's memory while it is being read?

```python
"""Watch the C tokenizer refill its buffer, one line at a time.

A debug build compiled with -d prints a line to stderr every time the tokenizer runs out of
input and asks its underflow function for more. Each of those lines is the whole of what the
tokenizer is holding at that moment, plus the value of tok->done, which is 10 for E_OK and 11
for E_EOF.

Two files go through it. The first parses cleanly. The second has an unclosed bracket on line
three, which is here to show that a failing parse reads the file more than once.
"""

import re
import subprocess
import sys
import tempfile
import time
from pathlib import Path

GOOD = "a = 1\nb = 2\nc = 3\nd = 4\ne = 5\n"
BAD = "a = 1\nb = 2\nc = ((\nd = 4\ne = 5\n"

#: The shape of the trace line, which is written by the fprintf in Parser/lexer/lexer.c.
TRACE = re.compile(r'^line\[(\d+)\] = "(.*)"  tok->done = (\d+)$')


def trace(source):
    """Run one file under -d and return the tokenizer's refill lines, in order.

    Once the parse of our own file is over, the interpreter goes on to compile other things
    while it builds the traceback, and those show up in the same trace. So the walk stops at
    the first refill whose text is not one of our own lines.
    """
    ours = {line + "\\n" for line in source.splitlines()} | {""}
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as handle:
        handle.write(source)
        path = handle.name
    said = subprocess.run([sys.executable, "-d", path], capture_output=True, text=True).stderr
    Path(path).unlink()
    found = []
    for line in said.splitlines():
        seen = TRACE.match(line)
        if not seen:
            continue
        if seen.group(2) not in ours:
            break
        found.append((int(seen.group(1)), seen.group(2), int(seen.group(3))))
    return found


started = time.monotonic()
for label, source in (("five lines that parse", GOOD), ("line 3 opens a bracket", BAD)):
    print(label)
    print()
    seen = trace(source)
    for number, text, done in seen:
        print(f'    line[{number}] = "{text}"  tok->done = {done}')
    print()
    print(f"    refills: {len(seen)}, for a file of {len(source.splitlines())} lines")
    print()
took = time.monotonic() - started

print(f"~ how long the two runs took, in seconds: {took:.1f}")
```

```text
five lines that parse

    line[1] = "a = 1\n"  tok->done = 10
    line[2] = "b = 2\n"  tok->done = 10
    line[3] = "c = 3\n"  tok->done = 10
    line[4] = "d = 4\n"  tok->done = 10
    line[5] = "e = 5\n"  tok->done = 10
    line[5] = ""  tok->done = 11

    refills: 6, for a file of 5 lines

line 3 opens a bracket

    line[1] = "a = 1\n"  tok->done = 10
    line[2] = "b = 2\n"  tok->done = 10
    line[3] = "c = ((\n"  tok->done = 10
    line[4] = "d = 4\n"  tok->done = 10
    line[5] = "e = 5\n"  tok->done = 10
    line[1] = "a = 1\n"  tok->done = 10
    line[2] = "b = 2\n"  tok->done = 10
    line[5] = ""  tok->done = 11
    line[1] = "a = 1\n"  tok->done = 10
    line[2] = "b = 2\n"  tok->done = 10

    refills: 10, for a file of 5 lines

~ how long the two runs took, in seconds: 0.7
```

That ran on Python 3.15.0rc1 in the debug build this project publishes, which is `ghcr.io/tamnd/cpython-internals/cpython:debug@sha256:de0d69b176872e99c63152b8ab4ebffac7bb131e5422c8eaebc7f455374ea824`. You do not need that build to read the numbers, and you do need it to produce them, which is why this is recorded rather than left as a cell you run. If you want to watch it happen yourself, `docker run --rm -i ghcr.io/tamnd/cpython-internals/cpython:debug@sha256:de0d69b176872e99c63152b8ab4ebffac7bb131e5422c8eaebc7f455374ea824 python3 -` takes the program on standard input.

Read the top block first. Five lines in, six refills out, and the last one is empty with `tok->done = 11`, which is `E_EOF` in [Include/errcode.h:23-24@v3.15.0rc1](https://github.com/python/cpython/blob/v3.15.0rc1/Include/errcode.h#L23-L24). `10` is `E_OK`, which is worth noticing on its own: these codes do not start at zero, because zero would be ambiguous with a character value.

Now the second block. Same file, one unclosed bracket, and the lines go past ten. A parse that fails runs again with a heavier set of grammar rules turned on so it can produce a better message, at [Parser/pegen.c:957-962@v3.15.0rc1](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/pegen.c#L957-L962), and there is a third walk in [Parser/pegen_errors.c:117-123@v3.15.0rc1](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/pegen_errors.c#L117-L123) that tokenizes the whole input looking for unclosed brackets specifically. F05 is about that machinery. For now the useful part is that a good error message costs a re-read, and CPython has decided that is a fine price for a file that was going to fail anyway.

## The whole memory of it

So if the buffer is one line, what carries across lines?

`struct tok_state` runs from [Parser/lexer/state.h:74-112@v3.15.0rc1](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/lexer/state.h#L74-L112) to line 140. That is about fifty fields, and most of them are plumbing: buffers, the encoding, the readline callable, the f-string mode stack. Eight of them are the actual state machine.

![the eight fields of struct tok_state that decide the token stream](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/f01-the-tokenizer-in-c/diagrams/the-state-that-matters.svg)

Two of those eight are fixed size arrays, and that is where Python's two least famous limits come from. [Parser/lexer/state.h:6-8@v3.15.0rc1](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/lexer/state.h#L6-L8) sets `MAXINDENT` to 100 and `MAXLEVEL` to 200, so the deepest you can indent and the deepest you can nest brackets are both fixed at compile time, and you can find both numbers from Python without reading the header.

In [ ]:
def deepest(build):
    """Grow the source until it stops compiling, and return the last size that worked."""
    size = 1
    while True:
        try:
            compile(build(size), "<here>", "exec")
        except SyntaxError:
            return size - 1
        size += 1


def indented(size):
    return "".join(" " * n + "if True:\n" for n in range(size)) + " " * size + "pass\n"


def nested(size):
    return "x = " + "(" * size + ")" * size + "\n"


print(f"deepest indentation that compiles:  {deepest(indented)}")
print(f"deepest bracket nesting:            {deepest(nested)}")

> **Version note.** These are the two array sizes in state.h, so an interpreter someone rebuilt with different limits will print different numbers. Every stock build agrees.

200 is `MAXLEVEL` exactly. 99 is `MAXINDENT` minus one, and the reason is a `+1` in the comparison at [Parser/lexer/lexer.c:582-586@v3.15.0rc1](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/lexer/lexer.c#L582-L586): the check runs before the push, so the hundredth level is refused rather than the hundred and first. Small thing, but this is the kind of off by one that a reimplementation gets wrong and only finds out about when someone's generated code stops compiling.

## The counter that owes you tokens

Here is the part T02 could see the shape of but not the reason for.

When a line is less indented than the last one, several blocks close at once, and several `DEDENT` tokens have to come out. But `tok_get` returns one token per call. So it cannot return three.

What it does instead is keep a counter. [Parser/lexer/lexer.c:571-609@v3.15.0rc1](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/lexer/lexer.c#L571-L609) is the indentation comparison, and in the dedent branch it pops the stack in a loop and does `tok->pendin--` on each pop. It has now not returned anything. Then on the next call, and the call after that, [Parser/lexer/lexer.c:616-633@v3.15.0rc1](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/lexer/lexer.c#L616-L633) sees a non zero `pendin`, moves it one step towards zero, and returns a single `DEDENT`.

![one line ending becoming two DEDENT tokens through the pendin counter](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/f01-the-tokenizer-in-c/diagrams/pendin-drains.svg)

The tell is in the positions. Those tokens are made without reading any characters, so every DEDENT from the same line ending reports the same position and an empty string, because they are drained from a counter rather than matched against text.

In [ ]:
SOURCE = "if a:\n    if b:\n        if c:\n            pass\nx = 1\n"

for found in tokenize.generate_tokens(io.StringIO(SOURCE).readline):
    if found.type in (token.INDENT, token.DEDENT):
        name = token.tok_name[found.type]
        print(f"  {name:7} start={found.start}  end={found.end}  string={found.string!r}")

Three `INDENT` tokens that each cover real spaces, and three `DEDENT` tokens that all sit at row 5 column 0 and cover nothing. That is `pendin` draining, seen from Python.

The empty string is worth one more note. In the parser's mode the tokenizer does not even fill in the position for these, and the `if (tok->tok_extra_tokens)` guard you can see in that block is what turns it on for `tokenize`. The parser does not need it. Your editor does.

## Two kinds of error

Last piece, and it is the one that makes the tokenizer's error messages make sense.

The tokenizer reports trouble two different ways. Sometimes it knows exactly what is wrong and calls [Parser/tokenizer/helpers.c:67-76@v3.15.0rc1#_PyTokenizer_syntaxerror](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/tokenizer/helpers.c#L67-L76) with the message written right there in the lexer. Sometimes it sets `tok->done` to a number from `errcode.h` and returns, and the words get chosen much later by a `switch` in [Parser/pegen_errors.c:34-73@v3.15.0rc1](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/pegen_errors.c#L34-L73).

![messages written in the lexer against error codes translated by the parser](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/f01-the-tokenizer-in-c/diagrams/two-kinds-of-error.svg)

some tokenizer errors carry their message from the lexer and others carry only a number that the parser turns into words, and which is which explains why a TabError and an unterminated string feel like they come from different places

In [ ]:
DEEP = "".join(" " * n + "if True:\n" for n in range(120)) + " " * 120 + "pass\n"

CASES = [
    ("an unclosed quote", "x = 'abc", "lexer.c"),
    ("a bad number", "x = 1_", "lexer.c"),
    ("201 open brackets", "x = " + "(" * 201, "lexer.c"),
    ("a tab after spaces", "if 1:\n\tif 1:\n        pass\n \tpass\n", "pegen_errors.c"),
    ("120 levels of indent", DEEP, "pegen_errors.c"),
    ("a dedent to nowhere", "if 1:\n    if 1:\n        pass\n      pass\n", "pegen_errors.c"),
    ("junk after a backslash", "x = 1 \\ 2\n", "pegen_errors.c"),
]

for label, source, written_in in CASES:
    try:
        compile(source, "<here>", "exec")
        said = "no error at all"
    except SyntaxError as unhappy:
        said = f"{type(unhappy).__name__}: {unhappy.msg}"
    print(f"  {label:24} {written_in:16} {said}")

The top three messages are string literals in `lexer.c`. The bottom four are not: the lexer set `E_TABSPACE`, `E_TOODEEP`, `E_DEDENT` and `E_LINECONT`, and every word you see was picked by that `switch`.

You can feel the difference in the wording. The lexer knows it was looking at a number, so it says so. The parser only has a number, so it says the general thing. That is also why `TabError` and `IndentationError` exist as separate exception types at all: the `switch` is the only place that gets to choose an exception class, and it chooses from four.

## Try it yourself

1. Add a line to `Grammar/Tokens` in a checkout, run `python Tools/build/generate_token.py all`, and read the diff. Four files should change and one of them is documentation.
2. Take the `deepest` helper and point it at f-string nesting instead. `MAXFSTRINGLEVEL` is in the same header. Do you get the number the header says, or one less, and why?
3. Find a source file where the tokenizer's own error is wrong about what you meant, then look up which of the two paths produced it.
4. `tokenize.generate_tokens` gives you `NL` where the parser sees nothing. Print both streams for a file with comments and blank lines, and count the difference.
5. Read [Parser/lexer/lexer.c:571-609@v3.15.0rc1](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/lexer/lexer.c#L571-L609) once, slowly. It is 39 lines and it is the whole of Python's indentation.

## What just happened

The token numbers are not written anywhere in the C. `Grammar/Tokens` is 78 lines of table and four files are generated from it, including the `token` module you import.

Four constructors get text into the tokenizer and they differ in one field, a function pointer called `underflow`. Which one you get depends on whether you passed `str` or `bytes` or a filename or a callable, and the `str` and `bytes` cases disagree about coding cookies.

The buffer holds one line. A file that parses is read once. A file that does not is read again, because a good error message is worth a second pass.

Everything that survives between lines lives in eight fields. Two of them are fixed arrays, which is where the 100 and the 200 come from, and one of them is a counter called `pendin` that explains why three `DEDENT` tokens all claim the same position.

Errors come out two ways: a message written in the lexer, or a number that the parser turns into words. Which one you got tells you where to look.

## Where this goes next

F02 stays in the same files and does f-strings, which since 3.12 are tokenized properly rather than pattern matched, and t-strings, which 3.14 added. That is the `tok_mode_stack` field this lesson skipped over.

After that the token stream stops being the subject and becomes the input. F03 is the PEG grammar, which is CPython's second machine readable table, and it is much bigger than this one.